In [ ]:
import os, subprocess, sys
open('/tmp/ddp_env.py','w').write('import os, torch, torch.distributed as dist\nlr = int(os.environ.get("LOCAL_RANK", "0"))\nprint("rank", lr, "CVD", os.environ.get("CUDA_VISIBLE_DEVICES"),\n      "count", torch.cuda.device_count(), flush=True)\ntry:\n    torch.cuda.set_device(lr)\n    print("rank", lr, "device", torch.cuda.get_device_name(lr), flush=True)\nexcept Exception as exc:\n    print("rank", lr, "set_device failed", exc, flush=True)\ntry:\n    dist.init_process_group("nccl")\n    t = torch.ones(1024, device=f"cuda:{lr}")\n    dist.all_reduce(t)\n    print("rank", lr, "all_reduce ok", float(t.sum()), flush=True)\n    dist.destroy_process_group()\nexcept Exception as exc:\n    print("rank", lr, "NCCL failed", type(exc).__name__, str(exc)[:300], flush=True)\n')
for label, extra in (('plain', {}), ('multi_rank', {'NCCL_MULTI_RANK_GPU_ENABLE':'1'})):
    env = dict(os.environ, **extra)
    r = subprocess.run(['torchrun','--nproc_per_node=2','--standalone','/tmp/ddp_env.py'],
                       capture_output=True, text=True, env=env, timeout=180)
    print('===', label, 'exit', r.returncode)
    print(r.stdout[-2000:])
    print(r.stderr[-1500:])